## Notebook Tutorial Summary

- Environment setup and API configuration
    - Loading environment variables with `dotenv`
    - Configuring OpenRouter credentials and base URLs

- TypeSafe classification fundamentals
    - Using `TypeSafeClassifier`
    - Evaluating structured questions with `Noul`, `Choice`, and `Score`
    - Interpreting classifier results and confidence scores

- Operational triage and incident routing
    - Determining whether an issue is urgent
    - Assigning the correct team (`infra` vs `billing`)
    - Scoring incident severity

- Model choice / routing middleware
    - Routing requests to different models based on task complexity
    - Example of `ModelRouterMiddleware`
    - Using model selection to optimize latency and cost

- Tool safety and risk gating
    - Protecting high-risk actions with `AutoModeMiddleware`
    - Preventing unsafe tool usage such as deleting all backups
    - Demonstrating LLM/tool guardrails

- LLM evaluation with “Jev as Judge”
    - Checking factual support against a source document
    - Detecting hallucination and measuring completeness
    - Using structured evaluation to judge model outputs

- PII and sensitive data detection
    - Identifying personally identifiable information (PII)
    - Checking for credentials and sensitive content
    - Applying threshold-based safeguards and human review policies

- Retrieval-augmented generation (RAG) reranking
    - Introducing Jev reranking for relevance scoring
    - Using `BM25Retriever` for candidate retrieval
    - Re-ranking retrieved passages with a custom `TypeSafeLegalReranker`

- Legal document matching and relevance scoring
    - Scoring whether a candidate passage matches a cited legal proposition
    - Building a document compression pipeline with `ContextualCompressionRetriever`
    - Ranking results by relevance and metadata

- End-to-end workflow patterns
    - Combining guardrails, evaluation, and retrieval in a single notebook workflow
    - Using LangChain primitives and TypeSafe judgment layers for production-style AI pipelines

In [32]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_typesafe import Choice, Noul, Score, TypeSafeClassifier

classifier = TypeSafeClassifier(api_key=os.environ.get("OPEN_ROUTER_API_KEY", "your_openrouter_api_key_here"),
    base_url="https://openrouter.ai/api")

/var/folders/57/y3hsdcy135b1wr5gzcw3b3kr0000gn/T/ipykernel_57321/710173286.py:3: LangChainBetaWarning: The class `TypeSafeClassifier` is in beta. It is actively being worked on, so the API may change.
  classifier = TypeSafeClassifier(api_key=os.environ.get("OPEN_ROUTER_API_KEY", "your_openrouter_api_key_here"),


In [3]:
response = classifier.invoke(
    {
        "state": (
            "The deploy failed twice and customers are seeing 500s. "
            "Can someone look now?"
        ),
        "questions": {
            "urgent": Noul(instructions="Does this need attention right now?"),
            "team": Choice(
                instructions="Which team should pick this up?",
                criteria={
                    "infra": "Deploys, availability, and on-call incidents.",
                    "billing": "Payments, invoices, and subscriptions.",
                },
            ),
            "severity": Score(
                instructions="How severe is the impact?",
                criteria=["Cosmetic.", "Degraded for some users.", "Full outage."],
            ),
        },
    }
)

print(response.nouls["urgent"].noul)
print(response.choices["team"].choice, response.choices["team"].confidence)
print(response.scores["severity"].score)

0.97
infra 1.0
1.63


In [5]:
print(response.scores)

{'severity': ScoreAnswer(type='score', score=1.63, legend={0: 'Cosmetic.', 1: 'Degraded for some users.', 2: 'Full outage.'}, probabilities={0: 0.0, 1: 0.37, 2: 0.63}, confidence=0.45)}


In [10]:
# The URL for the TypeSafe router making the model decision:
os.environ["TYPESAFE_BASE_URL"] = "https://openrouter.ai/api" 
os.environ["TYPESAFE_API_KEY"] = os.environ.get("OPEN_ROUTER_API_KEY", "your_openrouter_api_key_here")

### Model Choice

In [11]:
from langchain.agents import create_agent
from langchain_typesafe.experimental.middleware import (
    ModelChoice,
    ModelRouterMiddleware,
)

router = ModelRouterMiddleware(
    choices={
        "tier_1_support": ModelChoice(
            model="openai:gpt-5.6-terra",
            criteria="Standard FAQs, order tracking, password resets, and simple informational requests.",
        ),
        "tier_2_escalation": ModelChoice(
            model="openai:gpt-6-astra",
            criteria="Complex billing disputes, angry or churn-risk customers, and multi-step technical troubleshooting.",
        ),
    },
    instructions="Route customer queries to tier_1_support by default to minimize latency and cost. Escalate to tier_2_escalation only if the user is highly frustrated, threatens cancellation, or presents a multi-layered issue.",
)

agent = create_agent("openai:gpt-5.6-terra", middleware=[router])

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I've been charged three times this month for my subscription and your app keeps crashing when I open settings. Fix this immediately or I am canceling my account!",
            }
        ]
    }
)
print(result["model_route"].choice) # Expected output: tier_2_escalation

tier_2_escalation


### ToolRiskGating

In [12]:
from langchain.agents import create_agent
from langchain.messages import ToolMessage
from langchain.tools import tool
from langchain_typesafe import NoulCriteria
from langchain_typesafe.experimental.middleware import AutoModeMiddleware


@tool
def delete_all_backups() -> str:
    """Delete every backup. This action cannot be undone."""
    return "Backups deleted."


agent = create_agent(
    "openai:gpt-6-astra",
    tools=[delete_all_backups],
    middleware=[AutoModeMiddleware(tools=[delete_all_backups])],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all backups."}]}
)
print(result["messages"][-1].content)

[{'id': 'rs_0a727c65bfa69bd6006abf77ef660087d1a118bf016f303bc9', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqv3fxsm4FtOxt55BLnYd8UvuJyT0KRafQsgR7Tb94X4A5qIcWYoFzMdeeyYvXo45uzUAAr_qiQGRjlOk44lxQTeYtk3EkCDdxoxSUpaFDOJvkBcEhZVGbDt4KJNElVTdEs66LgJjb1OlEMBpBcBAlpLKTGwU8GrMhxnVQqQyRtjjaiHcc_K_6b0IoS5b45Wv57GzhSAi00RZBwGWo4iZnZ-TWvXpOO0nV7RwpC_MZI3Dz5mmsbCHCVRgC8A-nnPfwL4mW6m6-qDfYwfrHXc3yk10ARgq23hBck989fkzfSdKAgwDoj2WSqdSurDs7B6n-s3IcHSodTTvlvm9UE9g5FmX8SPgIqCJoNNqDaf8jGUuXNk-CF0MBdjeJVs2eKShs3m_ywVkcqbfrF7ZILdyZvzGPDFrjdqYf5zdyqx6AbgyGli8ILDYSTjASjFYexAWQwnTiY4C06VK3n5J2CwiShqEHzn5zouFfWIHpkpEGrNNmEUfAIbUTaQQpKFSdwBY38UTu6yLEYFYCg15a01kImQPADbWtaB4v2WWw0-8Cs3dBGqkK6bL4jI6NEY9V0ZPaUPdzSE36Hp9x_J0ELksn276waNsZxeSaDlsfh4dcjJ2waY9SjOC1x0J69i-AoMKT4oxyflJ8LFIKLb6ekygN0a2LVRT_pdC3nTX3txstVRBbTTfIa8GHUOfziQ_50oYzIckYwwXt-568aPXKczArLvKq0dvqRvJMMDjv1NtclAOtK6o1d6KZ9frV2XTNdRwor8RARajbOFMh9LjJZZRC4vrHsPzO83-E3VyN09Vyzvwdlxey8ZMb28JBDNBkX7dUz4gDhMRl-xN-xr8Y9BD9vJ

In [15]:
print(result["messages"][-1].content[1]['text'])

Deleting all backups is permanent and cannot be undone. Please confirm: do you want to delete every backup?


### Jev as Judge

In [16]:
from langchain_typesafe import TypeSafeClassifier, Noul, Score

# Initialize the Jev classifier runnable
classifier = TypeSafeClassifier(api_key=os.environ.get("OPEN_ROUTER_API_KEY", "your_openrouter_api_key_here"),
    base_url="https://openrouter.ai/api")

# 1. State can be a string, dict/JSON, or a list of LangChain Message objects
state = {
    "source_document": "The company's revenue grew by 15% in Q3.",
    "llm_response": "The company saw a 20% growth in Q3."
}

# 2. Define your criteria using LangChain's Jev primitives
questions = {
    "is_faithful": Noul(
        instructions="Is the LLM response fully supported by the facts in the source document without inventing new numbers?"
    ),
    "completeness": Score(
        instructions="Rate how completely the response addresses the facts.",
        # Score in LangChain takes an ordered list of string criteria
        criteria=["Poor", "Fair", "Good", "Excellent", "Perfect"] 
    )
}

# 3. Invoke the classifier 
result = classifier.invoke({
    "state": state,
    "questions": questions
})

# Access Noul properties via .nouls["key"].noul
if result.nouls["is_faithful"].noul < 0.50:
    print("Flagged for Hallucination! LLM response does not match source.")

# Access Score properties via .scores["key"].score
print(f"Completeness Score (0-4 scale): {result.scores['completeness'].score}")

Flagged for Hallucination! LLM response does not match source.
Completeness Score (0-4 scale): 0.3


### PII Detection

In [17]:
state = {
    "user_input": "My name is John Doe and my SSN is 000-00-0000. Can you help me?"
}

questions = {
    "medical": Noul(instructions="Does this text contain medical information?"),
    "pii": Noul(instructions="Does this text contain personally identifiable information?"),
    "credentials": Noul(instructions="Does this text contain API keys, passwords or credentials?")
}

result = classifier.invoke({
    "state": state,
    "questions": questions
})

# Extract the continuous probability (0.0 to 1.0)
pii_probability = result.nouls["pii"].noul

# Apply threshold logic
if pii_probability > 0.90:
    print("Action: Redact PII or block request.")
elif pii_probability > 0.60:
    print("Action: Send for human review.")
else:
    print("Action: Allow file safely.")

Action: Redact PII or block request.


In [18]:
# Extract the continuous probability (0.0 to 1.0)
credentials_probability = result.nouls["credentials"].noul

# Apply threshold logic
if credentials_probability > 0.90:
    print("Action: Redact credentials or block request.")
elif credentials_probability > 0.60:
    print("Action: Send for human review.")
else:
    print("Action: Allow file safely.")

Action: Allow file safely.


### RAG Reranking  

https://huggingface.co/blog/hotchpotch/introducing-jev-reranker

In [ ]:
# uv add jev-reranker

# from jev_reranker import JevReranker

# query = "How long do I have to return an online order to ACME Shop?"
# documents = [
#     "ACME Shop accepts online returns within 30 days of delivery.",
#     "For ACME Shop online orders, submit your return request within 30 days of receiving the item.",
#     "ACME Shop in-store purchases can be returned within 14 days of purchase.",
#     "ACME Shop products come with a one-year repair warranty.",
#     "FooBar Shop accepts online returns within 60 days of delivery.",
# ]

# os.environ["TYPESAFE_BASE_URL"] = "https://openrouter.ai/api"
# os.environ["TYPESAFE_API_KEY"] = os.environ.get("OPEN_ROUTER_API_KEY", "your_openrouter_api_key_here")

# reranker = JevReranker(
#     api_key=os.environ["TYPESAFE_API_KEY"],
#     endpoint=os.environ["TYPESAFE_BASE_URL"],
# )

# response = await reranker.a_relevance_rerank(query, documents, threshold=0.2)

# for item in response["results"]:
#     print(f"{item['score']:.2f}  {item['text']}")

In [48]:
#### LangChain integration with Jev for Reranking in RAG

In [ ]:
import os
from typing import Sequence, List
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import BaseDocumentCompressor
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_typesafe import TypeSafeClassifier, Noul

# -------------------------------------------------------------------
# 1. Define the Jev Question using the LangChain primitive
# We embed the true/false criteria directly into the rigorous instructions.
# -------------------------------------------------------------------
is_cited_source = Noul(
    instructions=(
        "The query excerpt comes from a US federal court opinion and was written "
        "immediately around a citation to a precedent; the citation itself has been "
        "removed. Could the candidate passage be from that cited precedent? "
        "TRUE: The candidate passage states or establishes the specific rule, standard, "
        "holding, or fact pattern that the query excerpt attributes to its removed citation. "
        "FALSE: The candidate passage is merely on a similar topic or doctrine; it does not "
        "supply the specific proposition the query excerpt relies on."
    )
)

# -------------------------------------------------------------------
# 2. Build the LangChain Document Compressor (The Reranker)
# -------------------------------------------------------------------
class TypeSafeLegalReranker(BaseDocumentCompressor):
    """Takes a shortlist of documents, scores them with Jev, and reranks them."""
    classifier: TypeSafeClassifier
    top_k: int = 10  # How many to keep after reranking

    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks=None,
    ) -> Sequence[Document]:
        
        # 1. Format the batch payload for Jev
        inputs = [
            {
                "state": {
                    "query_excerpt": query, 
                    "candidate_passage": doc.page_content
                },
                "questions": {"is_cited_source": is_cited_source}
            }
            for doc in documents
        ]
        
        # 2. Execute concurrently 
        results = self.classifier.batch(inputs)
        
        # 3. Apply the Noul score back to the LangChain Documents
        scored_docs = []
        for doc, result in zip(documents, results):
            noul_score = result.nouls["is_cited_source"].noul
            
            # Attach the score to metadata for transparency downstream
            doc.metadata["noul_score"] = noul_score
            scored_docs.append((noul_score, doc))
            
        # 4. Sort by highest Noul score and return the new top_k
        scored_docs.sort(key=lambda x: x[0], reverse=True)
        return [doc for score, doc in scored_docs[:self.top_k]]


# -------------------------------------------------------------------
# 3. Setup the Two-Step Pipeline (Fast Search + Re-rank)
# -------------------------------------------------------------------
def build_retrieval_pipeline(corpus_texts: List[str]):

    # Step 1: Fast Search (BM25)
    # We ask BM25 to return a wide net of 30 candidates
    bm25_retriever = BM25Retriever.from_texts(corpus_texts)
    bm25_retriever.k = 30 
    
    # Step 2: Reranking (TypeSafe Jev)
    # We instantiate our compressor, keeping the top 10 best answers
    jev_classifier = TypeSafeClassifier(api_key=os.environ.get("OPEN_ROUTER_API_KEY", "your_openrouter_api_key_here"),
    base_url="https://openrouter.ai/api")
    jev_reranker = TypeSafeLegalReranker(classifier=jev_classifier, top_k=10)

    # Step 3: Combine them into a single LangChain retriever
    pipeline = ContextualCompressionRetriever(
        base_compressor=jev_reranker, 
        base_retriever=bm25_retriever
    )
    
    return pipeline

In [47]:
# -------------------------------------------------------------------
# 4. Execute a Query
# -------------------------------------------------------------------

# Mock corpus of court opinions
mock_corpus = [
    "The court finds that prior restraint on publication is a violation of the First Amendment...",
    "In cases of severe national security threats, temporary injunctions on press may be upheld.",
    # ... thousands of other passages
]

pipeline = build_retrieval_pipeline(mock_corpus)

query = "The lower court erred in issuing an injunction against the newspaper, as established in [REDACTED]."

# This single call automatically runs BM25 -> fetches 30 docs -> Jev batches 30 scores -> returns top 10
final_documents = pipeline.invoke(query)

for rank, doc in enumerate(final_documents, 1):
    print(f"Rank {rank} | Noul Score: {doc.metadata['noul_score']:.2f}")
    print(f"{doc.page_content[:100]}...\n")

Rank 1 | Noul Score: 0.66
The court finds that prior restraint on publication is a violation of the First Amendment......

Rank 2 | Noul Score: 0.21
In cases of severe national security threats, temporary injunctions on press may be upheld....

